# Fred API

In [1]:
key = "637b953c119df6792e7edac10fc5dcac"

## SOFR - IORB

In [2]:
from fredapi import Fred # run pip install fredapi
import pandas as pd

symbol_1 = "SOFR"
symbol_2 = "IORB"

fred = Fred(api_key=key)
df1 = fred.get_series(symbol_1).to_frame(name=symbol_1)
df2 = fred.get_series(symbol_2).to_frame(name=symbol_2)
df = pd.merge(df1,df2, right_index=True, left_index=True)
df['Close'] = (df.iloc[:,0] - df.iloc[:,1]).round(6)
df.index.name = 'Date'
df.to_csv('SOFR_Minus_IORB.csv')

## WRESBAL

In [3]:
symbol = "WRESBAL"
fred = Fred(api_key=key)
df = fred.get_series(symbol).to_frame(name=symbol)

## HYG vs SPY

In [4]:
import numpy as np
import pandas as pd
from datetime import datetime
import time
import yfinance as yahooFinance
from dateutil import parser

class Yahoo_Finance:
    
    """
        Read data from Yahoo Finance API
        
        Candles
        Price History
    """
        
    def candles(self, symbol, period='max', interval='1d'):
        """ 
            Get historical candles
            
            Periods:   1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max
            Intervals: 1m, 2m, 5m, 15m, 30m, 60m, 90m, 1h, 1d, 5d, 1wk, 1mo, 3mo
        """
        ticker = yahooFinance.Ticker(symbol)
        df = ticker.history(period=period, interval=interval)
        df.reset_index(inplace=True)
        df.rename({'index':'Date'}, axis=1, inplace=True)
        
        if 'Datetime' in df.columns.to_list():
            df.rename({'Datetime':'Date'}, axis=1, inplace=True)

        df['Symbol'] = symbol
        df = df[['Symbol','Date','Open','High','Low','Close','Volume']].copy()
        return df
    
    def price(self, symbol, period='max', interval='1d'):
        """
            Get historical close price
        """
        ticker = yahooFinance.Ticker(symbol)
        df = ticker.history(period=period, interval=interval)
        df.reset_index(inplace=True)
        df.columns = [item.lower() for item in df.columns]
        df = df[['date','close']].copy()
        df.rename({'date':'Date','close':'Close'}, axis=1, inplace=True)
        df['Symbol'] = symbol
        return df
    
    
yf = Yahoo_Finance()
hyg = yf.price('HYG')
spy = yf.price('SPY')
hyg.set_index('Date',inplace=True)
spy.set_index('Date',inplace=True)
hyg['HYG_5d_chg'] = hyg.Close.pct_change(5)
hyg['HYG_10d_chg'] = hyg.Close.pct_change(10)
hyg['HYG_20d_chg'] = hyg.Close.pct_change(20)

spy['SPY_5d_chg'] = spy.Close.pct_change(5)
spy['SPY_10d_chg'] = spy.Close.pct_change(10)
spy['SPY_20d_chg'] = spy.Close.pct_change(20)
df = pd.merge(hyg,spy, right_index=True, left_index=True)
df['5d_Difference'] = df.HYG_5d_chg - df.SPY_5d_chg
df['10d_Difference'] = df.HYG_10d_chg - df.SPY_10d_chg
df['20d_Difference'] = df.HYG_20d_chg - df.SPY_20d_chg

df_export = df[['5d_Difference']].copy()
df_export.columns = ['Close']
df_export.to_csv('HYG_SPY_5d_Difference.csv')

df_export = df[['10d_Difference']].copy()
df_export.columns = ['Close']
df_export.to_csv('HYG_SPY_10d_Difference.csv')

df_export = df[['20d_Difference']].copy()
df_export.columns = ['Close']
df_export.to_csv('HYG_SPY_20d_Difference.csv')